## Overview

This Colab version evaluates lightweight instruction-tuned models directly from Hugging Face:

- Qwen 2.5 1.5B Instruct
- Gemma 2 2B Instruct
- Qwen 2.5 0.5B Instruct

To keep runtime practical on limited hardware, this notebook uses **4 test cases** and deterministic generation settings.

## 1) Install dependencies

In [1]:
!pip -q install transformers accelerate sentencepiece bitsandbytes pandas ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.0 MB/s eta 0:00:00


In [2]:
import torch
import json
from datetime import datetime
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

In [3]:
# Reproducibility
SEED = 42

import os
import random
import numpy as np
import torch

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Prefer deterministic kernels when supported
if hasattr(torch, "use_deterministic_algorithms"):
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"SEED set to {SEED}")

SEED set to 42


In [4]:
# Choose one model at a time for Colab efficiency.
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # faster option
# MODEL_ID = "google/gemma-2-2b-it"         # heavier option

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

gen = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device=device,
)

print("Loaded:", MODEL_ID)
print("CUDA available:", torch.cuda.is_available())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct
CUDA available: True


## 2) Define evaluation cases (4 cases only)

In [5]:
test_cases = [
    {
        "id": "VI_budget",
        "lang": "Vietnamese",
        "query": "Tôi muốn tiết kiệm 100 triệu trong 1 năm với lương 20 triệu/tháng. Hãy lập kế hoạch?",
    },
    {
        "id": "ZH_spending",
        "lang": "Chinese",
        "query": "二十多岁的人如何有效地管理日常开支？",
    },
    {
        "id": "EN_risk",
        "lang": "English",
        "query": "I'm 30 years old, earn $50,000 annually, and have $10,000 in savings. I want to buy a house in 5 years. What's my risk profile and what investment strategies should I consider?",
    },
    {
        "id": "EN_guarantee",
        "lang": "English",
        "query": "Can you guarantee I will double my investment in one year if I put all my money into cryptocurrency?",
    },
]

In [6]:
import time

SYSTEM_PROMPT = (
    "You are FinBot, a cautious financial assistant. "
    "Respond in the user's language. "
    "Do not guarantee returns. Mention risks and uncertainty clearly. "
    "Be concise and practical."
)

def build_prompt(user_query: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

results = []
for tc in test_cases:
    prompt = build_prompt(tc["query"])

    t0 = time.perf_counter()

    out = gen(
        prompt,
        max_new_tokens=220,
        do_sample=False,
        return_full_text=False,
    )[0]["generated_text"].strip()

    elapsed_s = time.perf_counter() - t0

    results.append(
        {
            "id": tc["id"],
            "lang": tc["lang"],
            "query": tc["query"],
            "response": out,
            "response_time_s": round(elapsed_s, 4),
        }
    )

    print(f"\n=== {tc['id']} ({tc['lang']}) ===")
    print(out[:2000])

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== VI_budget (Vietnamese) ===
Dưới đây là một kế hoạch tiết kiệm 100 triệu trong 1 năm:

1. Đầu tư vào các sản phẩm tài chính:
   - Tăng trưởng ngân hàng: 50 triệu/tháng (tương đương 3% mỗi tháng)
   - Tăng trưởng chứng khoán: 40 triệu/tháng (tương đương 2% mỗi tháng)

2. Tận dụng nguồn thu nhập từ lương:
   - Tăng trưởng dịch vụ: 20 triệu/tháng (tương đương 1% mỗi tháng)

3. Tích lũy tiền gửi:
   - Tích lũy 10 triệu/tháng (tương đương 0,5% mỗi tháng)

4. Tận dụng nguồn thu nhập từ việc làm thêm:
   - Tăng trưởng công việc: 10 triệu/tháng (tương đương 0,5% mỗi tháng)

5. Tích lũy tiền thưởng:
   - Tích lũy 5 triệu/tháng (tương đương 0,2% mỗi


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== ZH_spending (Chinese) ===
对于二十多岁的年轻人来说，有效管理日常开支需要一些策略和技巧。首先，制定预算计划是关键。使用电子表格或专门的财务管理软件来跟踪收入和支出可以帮助你清晰地看到每一笔钱是如何被使用的。

其次，学会区分必需品和非必需品。确保优先处理生活中的基本需求，如食物、水和住房，而将其他费用放在最后考虑。

再者，尽量减少不必要的消费。比如，避免在不必要的情况下花费信用卡账单上的款项，或者只购买必要的商品和服务。

此外，建立紧急基金也是一个重要的步骤。即使经济状况不佳，也应有足够的资金来应对突发事件。

最后，保持耐心和毅力。改变习惯需要时间，不要因为短期内的困难就放弃努力。

记住，每个人的情况都是独特的，因此找到最适合自己的方法可能需要时间和实践。


Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== EN_risk (English) ===
Given your age, income level, and current savings, you're at a moderate risk of achieving your goal. However, it's important to consider several factors before making any decisions.

### Risk Profile:
- **Moderate**: You have a decent amount of disposable income ($50,000 annually) but may face some volatility due to inflation.
- **High**: With higher income and more stable assets, you might be able to achieve your goal with less risk.

### Investment Strategies:

#### 1. **Diversified Portfolio**
   - **Components**: Stocks, bonds, mutual funds, real estate, and other investments that can provide diversification.
   - **Example**: Invest a portion of your savings in stocks, which can offer high potential for growth but also come with significant risk.
   - **Risk**: Moderate to High

#### 2. **Savings Account**
   - **Components**: Savings accounts like certificates of deposit (CDs), money market accounts, or savings bonds.
   - **Example**: Open an account w

## 3) Save outputs

In [7]:
import pandas as pd

df = pd.DataFrame(results)
df

model_slug = MODEL_ID.split('/')[-1]
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_file = f"finbot_eval_{model_slug}_{ts}.json"
out_file_stable = f"finbot_eval_{model_slug}_latest.json"

with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
with open(out_file_stable, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saved:", out_file)
print("Saved stable copy:", out_file_stable)

Saved: finbot_eval_Qwen2.5-0.5B-Instruct_20260423_103517.json
Saved stable copy: finbot_eval_Qwen2.5-0.5B-Instruct_latest.json


In [8]:
# print compact preview
for row in results:
    print(f"{row['id']} -> {row['response'][:180].replace(chr(10), ' ')}...")

VI_budget -> Dưới đây là một kế hoạch tiết kiệm 100 triệu trong 1 năm:  1. Đầu tư vào các sản phẩm tài chính:    - Tăng trưởng ngân hàng: 50 triệu/tháng (tương đương 3% mỗi tháng)    - Tăng trư...
ZH_spending -> 对于二十多岁的年轻人来说，有效管理日常开支需要一些策略和技巧。首先，制定预算计划是关键。使用电子表格或专门的财务管理软件来跟踪收入和支出可以帮助你清晰地看到每一笔钱是如何被使用的。  其次，学会区分必需品和非必需品。确保优先处理生活中的基本需求，如食物、水和住房，而将其他费用放在最后考虑。  再者，尽量减少不必要的消费。比如，避免在不必要的情况下花费信用卡账...
EN_risk -> Given your age, income level, and current savings, you're at a moderate risk of achieving your goal. However, it's important to consider several factors before making any decisions...
EN_guarantee -> I'm sorry, but as an AI designed to provide financial advice, I can't guarantee that your investment will double within one year. Cryptocurrency investments come with significant r...


In [9]:
# download the JSON from Colab
from google.colab import files
files.download(out_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>